# Fresh Start NBA — Notebook 1: Data Setup

Run this notebook first. It mounts Google Drive and loads your two core datasets.
Every other notebook depends on the paths set here.

**Upload to Google Drive before running:**
- `Fresh_Start_NBA_Colab/data/nba_data.csv`
- `Fresh_Start_NBA_Colab/data/historical_lines.csv`
- `Fresh_Start_NBA_Colab/models/` (all `.pkl` files — needed by notebooks 2 and 4)

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# ── Configure these paths once ──────────────────────────────────────────────
BASE_DIR   = Path('/content/drive/MyDrive/Fresh_Start_NBA_Colab')
DATA_DIR   = BASE_DIR / 'data'
MODELS_DIR = BASE_DIR / 'models'
OUT_DIR    = BASE_DIR / 'outputs'

OUT_DIR.mkdir(parents=True, exist_ok=True)

NBA_DATA_PATH   = DATA_DIR / 'nba_data.csv'
LINES_DATA_PATH = DATA_DIR / 'historical_lines.csv'

print('Paths configured.')
print(f'  nba_data:  {NBA_DATA_PATH}')
print(f'  lines:     {LINES_DATA_PATH}')
print(f'  models:    {MODELS_DIR}')
print(f'  outputs:   {OUT_DIR}')

## Load Datasets

In [ ]:
nba = pd.read_csv(NBA_DATA_PATH, low_memory=False)
nba['game_date'] = pd.to_datetime(nba['game_date'])
print(f'nba_data: {len(nba):,} rows  |  {nba["game_date"].min().date()} → {nba["game_date"].max().date()}')
nba.head(3)

In [ ]:
lines = pd.read_csv(LINES_DATA_PATH, low_memory=False)
lines['game_date'] = pd.to_datetime(lines['game_date'])
print(f'historical_lines: {len(lines):,} rows  |  {lines["game_date"].min().date()} → {lines["game_date"].max().date()}')
lines.head(3)

## Data Health Check

In [ ]:
print('=== nba_data.csv health check ===')
print(f'Rows:           {len(nba):,}')
print(f'Columns:        {len(nba.columns)}')
print(f'Unique players: {nba["player"].nunique():,}')
print(f'Date range:     {nba["game_date"].min().date()} to {nba["game_date"].max().date()}')
print(f'Seasons:        {sorted(nba["SEASON_YEAR"].unique())}')
print()

# Missing value summary for key stat columns
key_cols = ['pts', 'trb', 'ast', 'stl', 'blk', 'tov', 'mp', 'game_date', 'player', 'team']
missing = nba[key_cols].isnull().sum()
missing = missing[missing > 0]
if len(missing):
    print('Missing values in key columns:')
    print(missing)
else:
    print('No missing values in key columns.')

In [ ]:
print('=== historical_lines.csv health check ===')
print(f'Rows:             {len(lines):,}')
print(f'Unique props:     {lines["prop"].unique()}')
print(f'Date range:       {lines["game_date"].min().date()} to {lines["game_date"].max().date()}')
print(f'Unique players:   {lines["player"].nunique():,}')
print()

# Line distribution by prop type
print('Rows per prop type:')
print(lines['prop'].value_counts().to_string())

In [ ]:
# Games per season breakdown
print('=== Games per season ===')
by_season = nba.groupby('SEASON_YEAR').agg(
    rows=('pts', 'count'),
    unique_players=('player', 'nunique'),
    unique_games=('GAME_ID', 'nunique')
).reset_index()
print(by_season.to_string(index=False))

In [ ]:
# Confirm models directory has .pkl files
import os
pkl_files = list(MODELS_DIR.glob('*.pkl'))
json_files = list(MODELS_DIR.glob('*.json'))
print(f'Model .pkl files found: {len(pkl_files)}')
print(f'Model .json files found: {len(json_files)}')
for f in sorted(pkl_files):
    size_kb = f.stat().st_size / 1024
    print(f'  {f.name:<45} {size_kb:>8.1f} KB')

## All checks passed — you're ready to run the other notebooks.

Keep this notebook's kernel alive (or re-run it first) before running notebooks 2–5.
The `BASE_DIR`, `DATA_DIR`, `MODELS_DIR`, and `OUT_DIR` variables are reused across notebooks.